# Notebook 05: Multi-Reflection Decomposition

**Gap 4**: The paper leaves the double-reflection component (photons reflecting
off both disk planes) unquantified. This notebook uses NMF and supervised
decomposition to isolate and quantify each emission component.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.decomposition.separate_reflections import NMFDecomposer, plot_decomposition
from src.utils.physics import ENERGY_BINS
from src.utils.io import load_dataset_hdf5


In [ ]:
# Load warped disk spectra
X, y = load_dataset_hdf5('../data/simulated/demo_sweep.h5')
pol_fracs = X[:, :, 1]  # (N, N_energy) — polarization fraction only (non-negative)

# Fit NMF with 4 components (direct, inner_reflect, outer_reflect, double_reflect)
decomposer = NMFDecomposer(n_components=4)
decomposer.fit(pol_fracs)
print('NMF components shape:', decomposer.components_.shape)


In [ ]:
# Plot basis components
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['navy', 'firebrick', 'goldenrod', 'gray']
for i, (comp, label) in enumerate(zip(decomposer.components_, decomposer.component_labels)):
    ax.plot(np.log10(ENERGY_BINS), comp, color=colors[i], label=label, lw=2)
ax.set_xlabel('log(Energy/keV)')
ax.set_ylabel('NMF component (pol. fraction)')
ax.set_title('Emission component basis spectra from NMF decomposition')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/nmf_components.png', dpi=150)
plt.show()


In [ ]:
# Decompose a single spectrum and show mixing fractions
sample_idx = 0
coeffs = decomposer.decompose(pol_fracs[sample_idx])
total = coeffs.sum()

print(f'Mixing fractions for sample {sample_idx}:')
for label, c in zip(decomposer.component_labels, coeffs):
    print(f'  {label:20s}: {100*c/total:.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(decomposer.component_labels, 100 * coeffs / total,
       color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Fractional contribution [%]')
ax.set_title('Emission component fractions')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('../results/figures/mixing_fractions.png', dpi=150)
plt.show()
